# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² mass spectrometry dataset using the `mlcroissant` library. We'll walk through the process of loading metadata and records, examining the available record sets and fields, loading data into DataFrames, and performing exploratory data analysis (EDA).

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and print summary info
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a dataclass, not a dict
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the `dataset.metadata.record_sets` property to list all available record sets, each identified by its `@id`. For each record set, we display its fields and their `@id`s.

In [ ]:
# List all record sets and their fields
def summarize_record_sets(metadata):
    record_sets = getattr(metadata, 'record_sets', [])
    print(f"Found {len(record_sets)} record set(s).\n")
    for rs in record_sets:
        print(f"Record Set: {rs.name if hasattr(rs, 'name') else ''}  (@id={rs.id})")
        if hasattr(rs, 'fields') and rs.fields:
            for f in rs.fields:
                print(f"   Field: {f.name} (@id={f.id}), type: {f.data_type}")
        print()
    return record_sets

record_sets = summarize_record_sets(metadata)

# For demonstration, pick the first record set for sample data preview below
if len(record_sets):
    example_record_set_id = record_sets[0].id
    print(f"Example record set id: {example_record_set_id}")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Load the data for each record set
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns for record set @id={record_set_id}")

# Show the column names and preview for the first record set
if record_set_ids:
    primary_record_set = record_set_ids[0]
    print(f"\nFields for record set {primary_record_set}:")
    print(dataframes[primary_record_set].columns.tolist())
    dataframes[primary_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes example operations for one numeric field.

In [ ]:
# For the demo, select a sample numeric field from the available columns

# Let's inspect possible numeric fields (e.g. 'cr:field_Coverage_Percent', 'cr:field_MW')
primary_df = dataframes[primary_record_set]
numeric_candidates = [col for col in primary_df.columns if primary_df[col].dtype in ['float64', 'int64']]
if not numeric_candidates:
    # Try to infer fields that might be numeric
    numeric_candidates = [col for col in primary_df.columns if any(x in col.lower() for x in ['percent', 'mw', 'count', 'number', 'amount', 'abundance', 'score'])]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = primary_df.columns[0]  # fallback

print(f"Analyzing numeric field: {numeric_field}")

threshold = 10
filtered_df = primary_df[primary_df[numeric_field].astype(float) > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (showing first 5):")
print(filtered_df.head())

# Normalization
normalized_col = f"{numeric_field}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
print(f"\nNormalized {numeric_field} for filtered records (showing first 5):")
print(filtered_df[[numeric_field, normalized_col]].head())

# Grouping by another column if available (e.g., 'cr:field_Description')
possible_group_fields = [c for c in primary_df.columns if any(k in c.lower() for k in ['category', 'type', 'description', 'family', 'group']) and c != numeric_field]
if possible_group_fields:
    group_field = possible_group_fields[0]
    try:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped by {group_field} (showing first 5 groups):")
        print(grouped_df.head())
    except Exception as e:
        print(f"Group by {group_field} not possible due to error: {e}")
else:
    print("No suitable field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, histogram of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field].astype(float), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field} (filtered > {threshold})')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()


## 6. Conclusion
In this notebook, we've demonstrated how to use `mlcroissant` for loading, exploring, and preparing the FAIR² protein mass spectrometry dataset. We've explored the schema programmatically with entity `@id` fields, loaded data, performed basic filtering and normalization, and visualized a target numeric value. For deeper biological or domain-specific analysis, you should use domain knowledge to select fields and design further data cleaning and feature extraction steps.
